# Phase 2 — GPU and model environment
## Goal
The user confirmed non-commercial research terms and adult consent, and supplied three images. Official buffalo_l weights are now local. Both actual model sessions ran with CPU fallback disabled; profiling confirmed CUDA-only computational execution.

The dynamic detector spatial input and recognition batch axis required static batch-one contracts (640x640 / 112x112). This changes input metadata only, in memory; downloaded weights are unchanged. See the [official ORT shape helper](https://onnxruntime.ai/docs/tutorials/mobile/helpers/make-dynamic-shape-fixed.html). Phase 1 remains unchanged.

Three single images cannot support multi-image enrollment, independent genuine probes or threshold calibration. **NOT READY FOR UNSUPERVISED ATTENDANCE.**

## Setup

In [ ]:
from pathlib import Path
import sys, json, os, subprocess
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/phase2_verification.yaml').exists())
sys.path.insert(0,str(ROOT/'src'))
os.environ['MPLCONFIGDIR'] = str(ROOT/'.cache/matplotlib')
from face_verification import *
config = load_config(root=ROOT)
REPORTS = ROOT/'reports/phase2'
# Real faces are opt-in. Set these manually only after confirming each participant's consent.
RUN_REAL_FACES = False
ADULT_CONSENT = False
IDENTITY_CONSENT = False
PRIVATE_MANIFEST = ROOT/'data/private/consent_manifest.json'
print('Interpreter:',sys.executable)
print('Real-face workflow enabled:',RUN_REAL_FACES and ADULT_CONSENT and IDENTITY_CONSENT)


PRIVATE_INPUTS = Path(os.environ.get('FACEATTEND_PRIVATE_INPUTS', str(Path.home()/'AppData/Local/FaceAttend/phase2/private')))
# Explicit executor opt-in; ordinary Run All keeps all defaults false.
if os.environ.get('FACEATTEND_CONSENTED_SMOKE') == '1':
    supplied = json.loads((PRIVATE_INPUTS/'input_manifest.json').read_text())
    assert supplied['adult_identity_verification_consent_confirmed_by_user'] is True
    assert supplied['research_terms_confirmed_by_user'] is True
    RUN_REAL_FACES = ADULT_CONSENT = IDENTITY_CONSENT = True
print('Consented three-image smoke enabled:',RUN_REAL_FACES)


## Runtime versions and hardware
The existing PyTorch CUDA 12.6/cuDNN 9 DLLs are preloaded by importing torch. ORT 1.20.2 matches the [official CUDA 12/cuDNN 9 matrix](https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html). Newer CUDA 12.8/13 defaults were not installed. No Phase 1 package was upgraded.

In [ ]:
import torch, onnxruntime as ort, cv2, insightface, cryptography
print(json.dumps({'torch':torch.__version__,'cuda':torch.version.cuda,'cudnn':torch.backends.cudnn.version(),
    'onnxruntime':ort.__version__,'opencv':cv2.__version__,'insightface':insightface.__version__,
    'cryptography':cryptography.__version__,'gpu':torch.cuda.get_device_name(0),
    'available_providers':ort.get_available_providers()},indent=2))
assert 'CUDAExecutionProvider' in ort.get_available_providers()

## Fresh-process GPU smoke

In [ ]:
smoke = subprocess.run([sys.executable,str(ROOT/'scripts/phase2_gpu_smoke.py')],cwd=ROOT,capture_output=True,text=True,timeout=120)
print(smoke.stdout)
assert smoke.returncode == 0, smoke.stderr

## Model contract and acquisition gate
[Official model zoo](https://github.com/deepinsight/insightface/blob/master/python-package/docs/model_zoo.md): buffalo_l contains SCRFD-10GF and ResNet50@WebFace600K recognition. Only these two ONNX files are loaded. Age/gender, expression and liveness are excluded. InsightFace's [MIT code license and research-only weights](https://github.com/deepinsight/insightface#license) differ.

`initialize_models` never downloads. After both confirmations, use the explicit acquisition script documented in the Phase 2 guide. Effective CUDA provider checks plus `session.disable_cpu_ep_fallback=1` fail closed; ORT may list CPU in its provider registry despite disabling graph fallback. Computational profile events must all be CUDA.

In [ ]:
from face_verification.gpu import inspect_profile
try:
    models = initialize_models(config,profile=True)
    import numpy as np
    models.detector.detect(np.zeros((640,640,3),np.uint8),input_size=(640,640),max_num=0)
    models.recognizer.get_feat(np.zeros((112,112,3),np.uint8))
    model_smoke = {'status':'ACTUAL_MODEL_SYNTHETIC_INPUT_PASS','model_id':models.model_id,
        'profiles': [inspect_profile(m.session.end_profiling()) for m in (models.detector,models.recognizer)]}
except VerificationError as exc:
    model_smoke = {'status':exc.status.value,'message':str(exc),'actual_models_tested':False}
print(json.dumps(model_smoke,indent=2))

## Checks

In [ ]:
integrity = subprocess.run([sys.executable,str(ROOT/'scripts/phase2_snapshot.py'),'--check'],cwd=ROOT,capture_output=True,text=True,timeout=120)
print(integrity.stdout)
assert integrity.returncode == 0, integrity.stderr
print(json.loads((REPORTS/'model_manifest.json').read_text()))

## Next steps
Actual SCRFD/ArcFace CUDA profiles and hashes are recorded above and in the model manifest. Consent and image availability are confirmed. The images stay in a restricted LocalAppData folder outside OneDrive and this project. Select **Python (FaceAttend GPU)** manually in VS Code. The documented InsightFace GPU substitution leaves one expected dependency-metadata warning; both runtime distributions must never coexist.

Quality and capture sufficiency, rather than model acquisition, are the remaining blockers.

## Latest same-session cohort — CUDA proof

This section reads the executed ten-image retest artifact. The initial three-image experiment above is retained as history. User-supplied labels are represented by pseudonyms; no portraits or vectors are displayed. The user confirmed all ten captures are from one session. Enrollment uses photos 1/2/4 and 7/8/9; probes use 3/5 and 6/10, selected before score inspection. The encrypted enrollment smoke used a process-only key and removed all templates afterward.

These results establish implementation behavior, not a calibrated matching decision, independent-session accuracy, liveness, FMR/FNMR reliability or production readiness.

In [ ]:
batch = json.loads((REPORTS/'batch02_results.json').read_text())
print(json.dumps({'subjects':batch['subjects'],'photos':batch['input_count'],
    'capture_sessions':batch['capture_sessions'],'gpu_profiles':batch['gpu_profiles']},indent=2))
assert all(p['status']=='CUDA_CONFIRMED' for p in batch['gpu_profiles'].values())

## Latest single-subject cross-session diagnostic

This section reads `batch03_results.json`, produced by an actual fresh-process GPU run. The user supplied one subject's ten photos and confirmed every photo came from a separate session; exact dates were not supplied. Enrollment photos 1/2/7 and probe photos 3/4/5/6/8/9/10 were fixed before inspecting scores. Photos 6/10 fail the unchanged size/clipping gate. No rejected image is embedded.

The three-image encrypted enrollment, duplicate-ID rejection and deletion passed. Five valid probes have diagnostic genuine scores 0.529–0.835; ten comparisons with explicitly labeled prior controls have scores 0.022–0.268. Session independence is user-confirmed for the new subject; cross-subject chronology is unconfirmed. Comparisons share subjects/templates, so the counts are dependent evidence, not independent population trials. All operational verification attempts refuse with THRESHOLD_NOT_CALIBRATED. No production threshold, FMR/FNMR, liveness or attendance claim is issued.

Names, original images and vectors remain out of public notebook outputs. Test templates and the process-only key were removed. Earlier cohorts above are preserved as history.

In [ ]:
latest = json.loads((REPORTS/'batch03_results.json').read_text())
print(json.dumps({'gpu_profiles':latest['gpu_profiles'],'input_count':latest['input_count'],'sessions':latest['sessions'],'session_dates_available':latest['session_dates_available']},indent=2))